### Actividad 1: Análisis inicial de patients.csv

Antes de realizar la carga con tipos explícitos, se clasifican las columnas de `patients.csv` para determinar los `dtypes`:
* **Identificadores (Id):** La columna `Id` es un UUID (cadena alfanumérica única). Debe quedarse como `object` o cadena de texto, ya que cada valor es único y no tiene sentido agruparlo.
* **Fechas (Dates):** Las columnas `BIRTHDATE` y `DEATHDATE` deben parsearse directamente como `datetime64[ns]` para poder calcular edades o secuencias temporales.
* **Categóricas:** Columnas como `MARITAL` (estado civil), `RACE`, `ETHNICITY`, `GENDER`, `CITY` y `STATE` toman valores repetitivos de un conjunto cerrado. Se deben cargar como `category` para ahorrar memoria.

In [1]:
import pandas as pd

# 1. Carga ingenua (directamente desde el directorio actual)
df_patients = pd.read_csv("patients.csv")
df_encounters = pd.read_csv("encounters.csv")
df_observations = pd.read_csv("observations.csv")


# 2. Función de medición con deep=True en Megabytes
def medir_memoria(df, nombre):
    mb = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Consumo inicial de {nombre}: {mb:.2f} MB")
    return mb


mb_patients_antes = medir_memoria(df_patients, "patients.csv")
mb_encounters_antes = medir_memoria(df_encounters, "encounters.csv")
mb_observations_antes = medir_memoria(df_observations, "observations.csv")

Consumo inicial de patients.csv: 27.08 MB
Consumo inicial de encounters.csv: 1047.70 MB
Consumo inicial de observations.csv: 10130.82 MB


In [5]:
import pandas as pd

# 1. Carga de los datos ingenuos (solo observations para no saturar)
df_obs_ingenuo = pd.read_csv("observations.csv")
mem_antes = df_obs_ingenuo.memory_usage(deep=True) / (1024**2)

# 2. Definir el diccionario de tipos de datos optimizados
dtypes_optimizados = {
    "PATIENT": "category",
    "ENCOUNTER": "category",
    "CODE": "category",
    "DESCRIPTION": "category",
    "VALUE": "category",
    "UNITS": "category",
    "TYPE": "category",
}

# 3. Carga optimizada (parse_dates convierte directamente a formato de fecha)
df_obs_opt = pd.read_csv(
    "observations.csv", dtype=dtypes_optimizados, parse_dates=["DATE"]
)
mem_despues = df_obs_opt.memory_usage(deep=True) / (1024**2)

# 4. Calcular y mostrar la reducción
df_comparacion = pd.DataFrame({"MB Antes": mem_antes, "MB Después": mem_despues})
df_comparacion["% Reducción"] = 100 - (
    df_comparacion["MB Después"] / df_comparacion["MB Antes"] * 100
)

print(
    f"Reducción total de la tabla: {100 - (df_obs_opt.memory_usage(deep=True).sum() / df_obs_ingenuo.memory_usage(deep=True).sum() * 100):.2f}%"
)
display(df_comparacion)

Reducción total de la tabla: 85.75%


,MB Antes,MB Después,% Reducción
Index,0.000126,0.000126,0.000000
DATE,1160.318542,134.529686,88.405797
PATIENT,1429.377913,36.008662,97.480816
ENCOUNTER,1397.862713,142.345531,89.816916
CATEGORY,958.836643,958.836643,0.000000
CODE,932.362841,33.656453,96.390198
DESCRIPTION,1544.774858,33.668684,97.820480
VALUE,973.830465,70.975812,92.711687
UNITS,811.113940,16.820840,97.926205
TYPE,922.341214,16.816315,98.176779


| Columna | dtype antes | dtype después | MB antes | MB después | Qué se pierde |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **DATE** | `object` | `datetime64[ns]` | 1160.31 | 134.52 | **Cadena original:** Se pierde el formato de texto exacto del archivo CSV original. |
| **PATIENT** | `object` | `category` | 1429.37 | 36.00 | **Mutabilidad y Regex:** Se pierden las operaciones vectorizadas de texto directas (`.str`). |
| **ENCOUNTER** | `object` | `category` | 1397.86 | 142.34 | **Igual que PATIENT:** Cuesta poder computacional (CPU) construir el diccionario interno la primera vez. |
| **CATEGORY** | `object` | `object` | 958.83 | 958.83 | **Nada:** No se optimizó en este paso. Podría pasarse a `category` para ahorrar más memoria. |
| **CODE** | `object` | `category` | 932.36 | 33.65 | **Manipulación de sub-cadenas:** Imposibilidad de hacer recortes directos sin convertir de vuelta a `object`. |
| **DESCRIPTION** | `object` | `category` | 1544.77 | 33.66 | **Búsqueda de texto libre:** Buscar descripciones por expresiones regulares requiere conversión previa. |
| **VALUE** | `object` | `category` | 973.83 | 70.97 | **Matemáticas directas:** Se pierde la capacidad de hacer cálculos estadísticos directamente sin antes usar `to_numeric`. |
| **UNITS** | `object` | `category` | 811.11 | 16.82 | **Concatenación:** No se pueden añadir sufijos de forma masiva sin alterar las categorías base. |
| **TYPE** | `object` | `category` | 922.34 | 16.81 | **Flexibilidad de limpieza:** Restringe ediciones "al vuelo" de filas individuales. |

In [6]:
import pandas as pd

# 1. Definimos los diccionarios de tipos basados en nuestra clasificación
dtypes_patients = {
    "MARITAL": "category",
    "RACE": "category",
    "ETHNICITY": "category",
    "GENDER": "category",
    "CITY": "category",
    "STATE": "category",
    "COUNTY": "category",
}

dtypes_encounters = {
    "PATIENT": "category",
    "ENCOUNTERCLASS": "category",
    "CODE": "category",
    "DESCRIPTION": "category",
    "REASONCODE": "category",
    "REASONDESCRIPTION": "category",
}

# 2. Carga optimizada (forzando parse_dates y dtypes)
df_patients_opt = pd.read_csv(
    "patients.csv", dtype=dtypes_patients, parse_dates=["BIRTHDATE", "DEATHDATE"]
)
df_encounters_opt = pd.read_csv(
    "encounters.csv", dtype=dtypes_encounters, parse_dates=["START", "STOP"]
)
# Recargamos la de observations que hicimos en la Act 2 para tener el dato a la mano
dtypes_obs = {
    "PATIENT": "category",
    "ENCOUNTER": "category",
    "CODE": "category",
    "DESCRIPTION": "category",
    "VALUE": "category",
    "UNITS": "category",
    "TYPE": "category",
}
df_observations_opt = pd.read_csv(
    "observations.csv", dtype=dtypes_obs, parse_dates=["DATE"], low_memory=False
)


# 3. Calculamos la memoria optimizada (en MB)
def mem_mb(df):
    return df.memory_usage(deep=True).sum() / (1024**2)


# Valores iniciales que ya habíamos calculado antes
mem_inicial = {
    "patients.csv": 27.08,
    "encounters.csv": 1047.70,
    "observations.csv": 10130.82,
}

mem_optimizada = {
    "patients.csv": mem_mb(df_patients_opt),
    "encounters.csv": mem_mb(df_encounters_opt),
    "observations.csv": mem_mb(df_observations_opt),
}

# 4. Generamos la tabla de entrega de la Actividad 1
df_resumen_act1 = pd.DataFrame({"MB Antes": mem_inicial, "MB Después": mem_optimizada})
df_resumen_act1["% Reducción"] = 100 - (
    df_resumen_act1["MB Después"] / df_resumen_act1["MB Antes"] * 100
)

print("=== TABLA DE ENTREGA: ACTIVIDAD 1 ===")
display(df_resumen_act1.round(2))

=== TABLA DE ENTREGA: ACTIVIDAD 1 ===


,MB Antes,MB Después,% Reducción
patients.csv,27.08,16.96,37.38
encounters.csv,1047.70,505.25,51.78
observations.csv,10130.82,1443.66,85.75


In [4]:
import pandas as pd

# 0. Carga asegurando parseo de fechas
df_patients = pd.read_csv("patients.csv", parse_dates=["BIRTHDATE", "DEATHDATE"])
df_encounters = pd.read_csv("encounters.csv", parse_dates=["START", "STOP"])
df_observations = pd.read_csv("observations.csv", low_memory=False)

# Normalizar zonas horarias: quitar UTC para comparar directamente con BIRTHDATE y DEATHDATE
if df_encounters["START"].dt.tz is not None:
    df_encounters["START"] = df_encounters["START"].dt.tz_localize(None)
if df_encounters["STOP"].dt.tz is not None:
    df_encounters["STOP"] = df_encounters["STOP"].dt.tz_localize(None)

# --- 1. Valores faltantes ---
print("=== 1. VALORES FALTANTES (PORCENTAJES) ===")
for nombre, df in [
    ("patients.csv", df_patients),
    ("encounters.csv", df_encounters),
    ("observations.csv", df_observations),
]:
    print(f"\n--- {nombre} ---")
    nulos_pct = (df.isnull().sum() / len(df)) * 100
    display(nulos_pct[nulos_pct > 0].round(2).to_frame(name="% Faltantes"))

# --- 2. Duplicados en patients.csv ---
print("\n=== 2. DUPLICADOS EN PATIENTS.CSV ===")
id_col = "Id" if "Id" in df_patients.columns else df_patients.columns[0]
duplicados = df_patients[id_col].duplicated().sum()
print(f"Total de IDs duplicados en patients.csv: {duplicados}")

# --- 3. Coherencia Temporal (Cruce patients vs encounters) ---
print("\n=== 3. COHERENCIA TEMPORAL ===")
df_cruce = df_encounters.merge(
    df_patients[[id_col, "BIRTHDATE", "DEATHDATE"]],
    left_on="PATIENT",
    right_on=id_col,
    how="inner",
)

# Encuentro antes de nacer
encuentros_antes_nacimiento = df_cruce[df_cruce["START"] < df_cruce["BIRTHDATE"]]
print(f"Encuentros clínicos antes del nacimiento: {len(encuentros_antes_nacimiento)}")

# Encuentro después de morir
encuentros_despues_muerte = df_cruce[
    df_cruce["DEATHDATE"].notnull() & (df_cruce["START"] > df_cruce["DEATHDATE"])
]
print(f"Encuentros clínicos después de la defunción: {len(encuentros_despues_muerte)}")

# --- 4. Auditoría de VALUE en observations.csv ---
print("\n=== 4. ANÁLISIS DE LA COLUMNA VALUE (OBSERVATIONS) ===")
valores_numericos = pd.to_numeric(df_observations["VALUE"], errors="coerce")
mascara_no_numericos = valores_numericos.isna() & df_observations["VALUE"].notna()

total_obs = len(df_observations)
total_no_num = mascara_no_numericos.sum()
pct_no_num = (total_no_num / total_obs) * 100

print(f"Total de observaciones: {total_obs:,}")
print(f"Observaciones no numéricas: {total_no_num:,} ({pct_no_num:.2f}%)")

print("\nEjemplos de valores no numéricos y su tipo/descripción:")
columnas_relevantes = [
    c for c in ["DESCRIPTION", "VALUE", "UNITS", "TYPE"] if c in df_observations.columns
]
display(
    df_observations.loc[mascara_no_numericos, columnas_relevantes]
    .drop_duplicates()
    .head(10)
)

=== 1. VALORES FALTANTES (PORCENTAJES) ===

--- patients.csv ---


,% Faltantes
DEATHDATE,86.59
DRIVERS,16.63
PASSPORT,21.47
PREFIX,18.99
MIDDLE,19.71
SUFFIX,98.90
MAIDEN,73.10
MARITAL,31.90
FIPS,25.95



--- encounters.csv ---


,% Faltantes
REASONCODE,37.2
REASONDESCRIPTION,37.2



--- observations.csv ---


,% Faltantes
ENCOUNTER,3.54
CATEGORY,3.54
UNITS,27.29



=== 2. DUPLICADOS EN PATIENTS.CSV ===
Total de IDs duplicados en patients.csv: 0

=== 3. COHERENCIA TEMPORAL ===
Encuentros clínicos antes del nacimiento: 0
Encuentros clínicos después de la defunción: 3401

=== 4. ANÁLISIS DE LA COLUMNA VALUE (OBSERVATIONS) ===
Total de observaciones: 17,633,075
Observaciones no numéricas: 6,511,097 (36.93%)

Ejemplos de valores no numéricos y su tipo/descripción:


,DESCRIPTION,VALUE,UNITS,TYPE
21,Tobacco smoking status,Never smoked tobacco (finding),NaN,text
325,Sexual orientation,Heterosexual (finding),NaN,text
326,HIV status,not HIV positive,{nominal},text
327,Abuse Status [OMAHA],No signs/symptoms,{nominal},text
328,Housing status,Patient is homeless,{nominal},text
329,Are you covered by health insurance or some ot...,No,{nominal},text
376,Tobacco smoking status,Smokes tobacco daily (finding),NaN,text
511,Within the last year have you been afraid of ...,No,NaN,text
512,Do you feel physically and emotionally safe wh...,Yes,NaN,text
513,Are you a refugee,No,NaN,text


### 3. Auditoría de Calidad de Datos (Hallazgos)

**1. Análisis de Valores Faltantes**
* **patients.csv:** La alta tasa de valores nulos en `DEATHDATE` (86.59%) es un comportamiento lógico y esperado, ya que la gran mayoría de los pacientes en la simulación siguen vivos. Las ausencias en columnas como `SUFFIX` (98.90%) o `MAIDEN` (apellido de soltera, 73.10%) son normales debido a la naturaleza opcional de esos datos demográficos.
* **encounters.csv:** Existe un 37.2% de valores faltantes en `REASONCODE` y `REASONDESCRIPTION`. Esto sugiere que no todas las visitas médicas tienen un código de diagnóstico principal explícito (pueden ser visitas administrativas, check-ups de rutina o seguimientos donde no se emite un nuevo diagnóstico).
* **observations.csv:** Destaca un 27.29% de falta de datos en la columna `UNITS`. Este faltante tiene sentido y está correlacionado con los resultados del punto 4, ya que las mediciones de tipo categórico o cualitativo no llevan unidades físicas.

**2. Duplicidad de Identificadores**
* El dataset tiene una integridad referencial limpia en su tabla principal: se encontraron **0 duplicados** en la llave primaria de `patients.csv`.

**3. Coherencia Temporal (Anomalías Lógicas)**
* **Positivo:** No existen registros de encuentros médicos anteriores a la fecha de nacimiento.
* **Anomalía explicada:** Se detectaron **3,401 encuentros clínicos posteriores a la fecha de defunción** del paciente. Aunque parece un error, en las bases de datos de salud (y modelado por Synthea) esto representa actividad administrativa post-mortem legítima, como la emisión de certificados de defunción, trámites de facturación, o reportes de autopsias.

**4. Naturaleza de la columna VALUE en observations.csv**
* De un total de 17,633,075 observaciones, el **36.93% (6,511,097 registros) no son convertibles a formato numérico**.
* **Diagnóstico:** Al inspeccionar estas filas, confirmamos que la columna `VALUE` está "sobrecargada". Almacena simultáneamente mediciones continuas (que sí son números) y respuestas de encuestas sociales o categóricas (ej. estado de tabaquismo, orientación sexual, estatus de vivienda).
* **Conclusión:** Forzar la conversión de esta columna a tipos de datos numéricos flotantes (`float32` o `float64`) para ahorrar memoria provocaría una pérdida de información destructiva, eliminando más de un tercio del historial clínico cualitativo de los pacientes.

### Actividad 4: Uniones y Validación de Cardinalidades

**Predicción del Merge 1: observations → encounters**
* **Cardinalidad esperada:** Muchos a uno (`m:1`). En una sola visita médica (encuentro) se le pueden hacer múltiples observaciones a un paciente (ej. medir peso, altura y presión arterial). Por lo tanto, muchas observaciones apuntan a un solo registro de encuentro.
* **Filas esperadas:** El resultado debe tener exactamente el mismo número de filas que la tabla `observations` original. Ninguna observación debe duplicarse.

**Predicción del Merge 2: tabla combinada → patients**
* **Cardinalidad esperada:** Muchos a uno (`m:1`). Un paciente a lo largo de su vida genera miles de observaciones clínicas en múltiples encuentros. Muchas filas del lado izquierdo apuntarán a un único registro maestro de paciente en el lado derecho.
* **Filas esperadas:** Exactamente el mismo número que resultó del Merge 1. No deben generarse filas extra.

In [7]:
# --- MERGE 1: observations -> encounters ---
print("=== MERGE 1: observations -> encounters ===")
print(f"Filas iniciales de observations: {len(df_observations_opt):,}")

# Identificamos el nombre real de la llave primaria en encounters (suele ser 'Id')
id_enc = "Id" if "Id" in df_encounters_opt.columns else df_encounters_opt.columns[0]

# Hacemos el cruce (Left Join)
df_obs_enc = df_observations_opt.merge(
    df_encounters_opt,
    left_on="ENCOUNTER",
    right_on=id_enc,
    how="left",
    suffixes=("_obs", "_enc"),  # Diferenciamos columnas repetidas
    validate="m:1",  # Rúbrica: Validamos la cardinalidad Muchos a Uno
    indicator=True,  # Rúbrica: Encendemos la auditoría
)

print(f"Filas después del primer merge: {len(df_obs_enc):,}")
print("\nAuditoría de unión (indicator):")
print(df_obs_enc["_merge"].value_counts())

# Quitamos la columna _merge para que no estorbe en el siguiente cruce
df_obs_enc = df_obs_enc.drop(columns=["_merge"])

# --- MERGE 2: tabla_combinada -> patients ---
print("\n\n=== MERGE 2: tabla combinada -> patients ===")

# Identificamos la llave primaria en patients
id_pat = "Id" if "Id" in df_patients_opt.columns else df_patients_opt.columns[0]

# Ojo: la llave foránea ahora se llama 'PATIENT_obs' por el sufijo que pusimos arriba
df_final = df_obs_enc.merge(
    df_patients_opt,
    left_on="PATIENT_obs",
    right_on=id_pat,
    how="left",
    validate="m:1",
    indicator=True,
)

print(f"Filas después del segundo merge: {len(df_final):,}")
print("\nAuditoría de unión (indicator):")
print(df_final["_merge"].value_counts())

=== MERGE 1: observations -> encounters ===
Filas iniciales de observations: 17,633,075
Filas después del primer merge: 17,633,075

Auditoría de unión (indicator):
_merge
both          17009564
left_only       623511
right_only           0
Name: count, dtype: int64


=== MERGE 2: tabla combinada -> patients ===
Filas después del segundo merge: 17,633,075

Auditoría de unión (indicator):
_merge
both          17633075
left_only            0
right_only           0
Name: count, dtype: int64


### 4. Auditoría de Unión (Resultados)

**Merge 1: observations → encounters**
* **Validación de cardinalidad:** Pasó exitosamente (`m:1`). Las filas se mantuvieron constantes en 17,633,075, confirmando que la relación era correcta y no hubo un producto cartesiano (multiplicación de filas).
* **Auditoría (indicator):** Se detectaron 623,511 registros `left_only` (que no cruzaron). Esto no es un error de unión, sino que corresponde exactamente al **3.54% de nulos** en la columna `ENCOUNTER` reportados en la Actividad 3 (observaciones socioeconómicas no ligadas a una visita médica).

**Merge 2: tabla combinada → patients**
* **Validación de cardinalidad:** Pasó exitosamente (`m:1`), manteniendo el mismo número de filas.
* **Auditoría (indicator):** El cruce fue 100% perfecto. Todos los 17,633,075 registros aparecen como `both`, lo que garantiza la integridad referencial absoluta: no existen observaciones en el sistema que carezcan de un paciente asignado.

In [8]:
# --- ACTIVIDAD 5: PREGUNTAS CLÍNICAS (PARTE 1) ---

# 1. Pacientes por grupo étnico y sexo
print("=== 1. PACIENTES POR GRUPO ÉTNICO Y SEXO ===")
# Ojo: Usamos drop_duplicates en el ID del paciente para no contar a la misma persona
# miles de veces por cada observación que tiene.
df_pacientes_unicos = df_final.drop_duplicates(subset=["PATIENT_obs"])

# Agrupamos por raza (RACE) y género (GENDER).
# La justificación de estos grupos la redactaremos en Markdown después.
tabla_demografia = (
    df_pacientes_unicos.groupby(["RACE", "GENDER"]).size().unstack(fill_value=0)
)
display(tabla_demografia)

# 2. Número medio y mediano de encuentros por paciente
print("\n=== 2. NÚMERO DE ENCUENTROS POR PACIENTE ===")
# Contamos cuántos encuentros ÚNICOS tiene cada paciente
encuentros_por_paciente = df_final.groupby("PATIENT_obs")["ENCOUNTER"].nunique()

media_encuentros = encuentros_por_paciente.mean()
mediana_encuentros = encuentros_por_paciente.median()

print(f"Media (promedio) de encuentros por paciente: {media_encuentros:.2f}")
print(f"Mediana de encuentros por paciente: {mediana_encuentros:.2f}")

# 3. Los 10 códigos de observación más frecuentes
print("\n=== 3. TOP 10 OBSERVACIONES MÁS FRECUENTES ===")
# value_counts sobre las columnas de código y descripción
top_10_obs = (
    df_final[["CODE_obs", "DESCRIPTION_obs"]]
    .value_counts()
    .head(10)
    .reset_index(name="Frecuencia")
)
display(top_10_obs)

=== 1. PACIENTES POR GRUPO ÉTNICO Y SEXO ===


GENDER,F,M
RACE,,
asian,845,812
black,919,976
hawaiian,113,110
native,61,49
other,137,154
white,9381,9540



=== 2. NÚMERO DE ENCUENTROS POR PACIENTE ===
Media (promedio) de encuentros por paciente: 31.49
Mediana de encuentros por paciente: 13.00

=== 3. TOP 10 OBSERVACIONES MÁS FRECUENTES ===


,CODE_obs,DESCRIPTION_obs,Frecuencia
0,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,554413
1,8462-4,Diastolic Blood Pressure,335571
2,8480-6,Systolic Blood Pressure,335571
3,29463-7,Body Weight,317917
4,8867-4,Heart rate,309777
5,9279-1,Respiratory rate,309777
6,8302-2,Body Height,302874
7,72166-2,Tobacco smoking status,302018
8,39156-5,Body mass index (BMI) [Ratio],281293
9,33914-3,Glomerular filtration rate [Volume Rate/Area] ...,242043


### 5. Análisis Clínico y Justificaciones

**1. Justificación de grupos demográficos:**
Se agrupó a los pacientes utilizando `RACE` y `GENDER`. Esta división permite observar los sesgos inherentes de la muestra sintética (predominantemente caucásica, reflejando el censo base de Synthea). No se utilizó `ETHNICITY` como agrupación principal porque en este estándar suele ser una variable binaria (Hispano/No Hispano), lo que oculta la diversidad poblacional real.

**2. Decisión: Media Global vs Media de Medias:**
Para responder a preguntas clínicas poblacionales, **se debe utilizar la Media de Medias**. 
*Justificación:* El gran cambio entre la media (31.49) y la mediana (13) de encuentros demuestra que una minoría de pacientes crónicos acude al hospital con extrema frecuencia. Si calculamos la media global de una prueba de laboratorio, esos pacientes crónicos aportarán cientos de mediciones, sesgando el promedio hacia la patología. Al calcular primero el promedio por paciente, y luego promediar la población general, le otorgamos el mismo peso estadístico a un individuo sano que a uno crónico.

In [9]:
import pandas as pd

print("=== ACTIVIDAD 6: FORMATO ANCHO Y VALORES IMPLAUSIBLES ===")

# Seleccionamos 3 analitos del Top 10 para nuestro formato ancho
# 8480-6 (Sistólica), 8462-4 (Diastólica), 8867-4 (Frecuencia cardíaca)
df_vitales = df_final[df_final["CODE_obs"].isin(["8480-6", "8462-4", "8867-4"])].copy()

# Aseguramos formato numérico
df_vitales["VALUE_num"] = pd.to_numeric(df_vitales["VALUE"], errors="coerce")
df_vitales = df_vitales.dropna(subset=["VALUE_num"])

# 1. Transformación a formato ancho (Matriz)
# Filas: Paciente y Fecha
# Columnas: El nombre del analito (DESCRIPTION_obs)
# Valores: La medición numérica
df_ancho = df_vitales.pivot_table(
    index=["PATIENT_obs", "DATE"],
    columns="DESCRIPTION_obs",
    values="VALUE_num",
    aggfunc="mean",  # Esta es la función que promedia silenciosamente los duplicados
).reset_index()

print("Muestra del DataFrame en formato ancho:")
display(df_ancho.head())

# 2. Búsqueda de valores implausibles (Límites fisiológicos)
col_sistolica = "Systolic Blood Pressure"
col_hr = "Heart rate"

# Filtramos valores incompatibles con la vida o rangos ambulatorios reales
implausibles_sistolica = df_ancho[
    (df_ancho[col_sistolica] < 40) | (df_ancho[col_sistolica] > 260)
]
implausibles_hr = df_ancho[(df_ancho[col_hr] < 20) | (df_ancho[col_hr] > 250)]

print(f"\nRegistros con Sistólica implausible: {len(implausibles_sistolica)}")
print(f"Registros con Frecuencia Cardíaca implausible: {len(implausibles_hr)}")

=== ACTIVIDAD 6: FORMATO ANCHO Y VALORES IMPLAUSIBLES ===
Muestra del DataFrame en formato ancho:


DESCRIPTION_obs,PATIENT_obs,DATE,Diastolic Blood Pressure,Heart rate,Systolic Blood Pressure
0,0004864b-3998-1959-0f78-458d77141a01,2017-07-05 02:43:31+00:00,83.0,91.0,134.0
1,0004864b-3998-1959-0f78-458d77141a01,2018-07-11 02:43:31+00:00,78.0,80.0,138.0
2,0004864b-3998-1959-0f78-458d77141a01,2019-06-19 02:43:31+00:00,79.0,94.0,137.0
3,0004864b-3998-1959-0f78-458d77141a01,2019-07-17 02:43:31+00:00,79.0,91.0,132.0
4,0004864b-3998-1959-0f78-458d77141a01,2020-01-15 02:43:31+00:00,86.0,62.0,132.0



Registros con Sistólica implausible: 0
Registros con Frecuencia Cardíaca implausible: 0


### 6. Formato Ancho y Control de Valores Implausibles

**1. El comportamiento de `pivot_table` con medidas repetidas:**
Al reestructurar el dataset a formato ancho (paciente y fecha en filas, analitos en columnas), surge un problema de cardinalidad: un paciente puede tener mediciones matutinas y vespertinas el mismo día. La función `pivot_table` de Pandas resuelve este conflicto calculando automáticamente la **media** (`aggfunc='mean'`) de los valores de ese día. Esto es útil, pero aplana la varianza intradiaria, lo cual ocultaría picos hipertensivos nocturnos.

**2. Detección de Valores Implausibles:**
Se evaluaron los límites compatibles con parámetros fisiológicos reales.
* **Presión Sistólica:** Se definió como implausible cualquier valor `< 40 mmHg` o `> 260 mmHg`. *(Referencia: Guyton & Hall, Tratado de Fisiología Médica. Valores fuera de este rango en un entorno no crítico de UCI son incompatibles con la vida o indican un error del esfigmomanómetro).*
* **Frecuencia Cardíaca:** Se definió como implausible `< 20 lpm` o `> 250 lpm`. *(Referencia: AHA Guidelines for CPR and ECC).*

**3. Acción correctiva (¿Qué hacer con ellos?):**
En este dataset sintético (Synthea) los valores suelen estar dentro de límites estrictos, por lo que los conteos de implausibles son frecuentemente cero. Sin embargo, en un entorno de datos de mundo real (RWD), la política recomendada no es eliminar la fila entera, ya que perderíamos las observaciones válidas de los otros analitos de ese día. Lo correcto es forzar la celda individual del valor anómalo a `NaN` (para excluirla de estadísticos descriptivos) y levantar una bandera (*flag*) de error de captura en el sistema.

In [10]:
import tracemalloc

import pandas as pd

print("=== ACTIVIDAD 7: PROCESAMIENTO POR LOTES (CHUNKS) ===")

# Iniciamos el rastreador de memoria de Python
tracemalloc.start()

# Definimos el tamaño del lote: 250,000 filas a la vez.
# Esto mantendrá el consumo de RAM muy por debajo del límite de 200 MB.
chunk_size = 250000

# Series para ir acumulando los resultados parciales de cada lote
suma_global = pd.Series(dtype="float64")
conteo_global = pd.Series(dtype="int64")

# Leemos SOLO las columnas necesarias del CSV grande para ahorrar aún más memoria
iterador_csv = pd.read_csv(
    "observations.csv", usecols=["DESCRIPTION", "VALUE"], chunksize=chunk_size
)

for i, chunk in enumerate(iterador_csv):
    # Intentamos convertir VALUE a numérico. Lo que es texto (categorías) se vuelve NaN
    chunk["VALUE_num"] = pd.to_numeric(chunk["VALUE"], errors="coerce")
    chunk_limpio = chunk.dropna(subset=["VALUE_num"])

    # Calculamos la suma y el conteo de este lote específico
    suma_lote = chunk_limpio.groupby("DESCRIPTION")["VALUE_num"].sum()
    conteo_lote = chunk_limpio.groupby("DESCRIPTION")["VALUE_num"].count()

    # Sumamos los resultados de este lote al acumulador global
    suma_global = suma_global.add(suma_lote, fill_value=0)
    conteo_global = conteo_global.add(conteo_lote, fill_value=0)

    # Imprimimos progreso para no quedarnos a ciegas
    if (i + 1) % 10 == 0:
        print(f"Procesados {i + 1} lotes (aprox {(i + 1) * chunk_size:,} filas)...")

# Calculamos la media global final dividiendo las sumas totales entre los conteos totales
media_global = suma_global / conteo_global

# Detenemos el rastreador y obtenemos el pico de memoria
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("\n--- RESULTADOS FINALES ---")
res_final = (
    pd.DataFrame({"Conteo Total": conteo_global, "Media Global": media_global})
    .sort_values("Conteo Total", ascending=False)
    .head(5)
)
display(res_final)

print(f"\nPico de memoria RAM utilizado: {peak / (1024**2):.2f} MB")

=== ACTIVIDAD 7: PROCESAMIENTO POR LOTES (CHUNKS) ===
Procesados 10 lotes (aprox 2,500,000 filas)...
Procesados 20 lotes (aprox 5,000,000 filas)...
Procesados 30 lotes (aprox 7,500,000 filas)...
Procesados 40 lotes (aprox 10,000,000 filas)...
Procesados 50 lotes (aprox 12,500,000 filas)...
Procesados 60 lotes (aprox 15,000,000 filas)...
Procesados 70 lotes (aprox 17,500,000 filas)...

--- RESULTADOS FINALES ---


,Conteo Total,Media Global
DESCRIPTION,,
Pain severity - 0-10 verbal numeric rating [Score] - Reported,554413.0,3.066943
Systolic Blood Pressure,335571.0,116.580413
Diastolic Blood Pressure,335571.0,78.008579
Body Weight,317917.0,70.654481
Heart rate,309777.0,80.969510



Pico de memoria RAM utilizado: 26.53 MB


### 7. Procesamiento Masivo por Lotes (Chunking)

**Presupuesto artificial:** 200 MB
**Pico de memoria real medido:** 26.53 MB

**Justificación del enfoque:**
El método ingenuo de `read_csv` intenta cargar el archivo completo en la memoria, lo cual resulta en ~10 GB de consumo debido al overhead de los objetos nativos de Python y los punteros. 

El enfoque por lotes (*chunking*) funciona y respeta el presupuesto estricto de memoria porque:
1. **Paginación:** Solo se cargan 250,000 líneas a la vez en la memoria.
2. **Reducción por Agregación:** En lugar de guardar los datos crudos, se extrae inmediatamente el valor matemático deseado (suma y conteo) agrupado por descripción. 
3. **Liberación de Memoria:** Al pasar al siguiente iterador, el bloque procesado anterior pierde sus referencias y el recolector de basura (*garbage collector*) de Python libera esa RAM inmediatamente.
La única estructura que crece constantemente en memoria es la tabla de resultados finales (agrupada por analito), que apenas ocupa unos pocos kilobytes.

In [11]:
!pip install polars pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 70.9 MB/s  0:00:05:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-4.2.0-py2.py3-none-any.whl size=450798675 sha256=924d324a386f9edf5a92b76003c5e1bc92374209633d3ced3c02bee86b19bbc6
  Stored in directory: /home/jorge/.cache/pip/wheels/4f/6d/69/496f1f5964e3470c76313fe2a499eb219a9c95ac1f1cb880ab
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


In [12]:
import time
import tracemalloc

import pandas as pd
import polars as pl
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

print("=== ACTIVIDAD 8: BENCHMARK DE MOTORES ===")

# --- 1. PANDAS ---
print("\n[Iniciando Pandas...]")
tracemalloc.start()
t0 = time.time()

df_p_pd = pd.read_csv("patients.csv", usecols=["Id", "RACE", "GENDER"])
df_e_pd = pd.read_csv("encounters.csv", usecols=["Id", "PATIENT"])
df_o_pd = pd.read_csv(
    "observations.csv", usecols=["PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"]
)

df_merged_pd = df_o_pd.merge(df_e_pd, left_on="ENCOUNTER", right_on="Id", how="inner")
df_merged_pd = df_merged_pd.merge(
    df_p_pd, left_on="PATIENT_x", right_on="Id", how="inner"
)
top10_pd = df_merged_pd["DESCRIPTION"].value_counts().head(10)

t1 = time.time()
_, peak_pd = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Pandas finalizado. Tiempo: {t1 - t0:.2f} s | RAM: {peak_pd / 1024**2:.2f} MB")


# --- 2. POLARS ---
print("\n[Iniciando Polars...]")
tracemalloc.start()
t0 = time.time()

# Polars usa 'scan_csv' (Evaluación perezosa) y optimiza la memoria nativamente
df_p_pl = pl.scan_csv("patients.csv").select(["Id", "RACE", "GENDER"])
df_e_pl = pl.scan_csv("encounters.csv").select(["Id", "PATIENT"])
df_o_pl = pl.scan_csv("observations.csv", ignore_errors=True).select(
    ["PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"]
)

df_merged_pl = df_o_pl.join(df_e_pl, left_on="ENCOUNTER", right_on="Id", how="inner")
df_merged_pl = df_merged_pl.join(df_p_pl, left_on="PATIENT", right_on="Id", how="inner")

top10_pl = (
    df_merged_pl.group_by("DESCRIPTION")
    .count()
    .sort("count", descending=True)
    .limit(10)
    .collect()
)

t1 = time.time()
_, peak_pl = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(
    f"Polars finalizado. Tiempo: {t1 - t0:.2f} s | RAM (Python driver): {peak_pl / 1024**2:.2f} MB"
)


# --- 3. PYSPARK ---
print("\n[Iniciando PySpark...]")
# Arrancamos la máquina virtual de Spark local
spark = SparkSession.builder.appName("Lab").master("local[*]").getOrCreate()

tracemalloc.start()
t0 = time.time()

df_p_sp = spark.read.csv("patients.csv", header=True).select("Id", "RACE", "GENDER")
df_e_sp = spark.read.csv("encounters.csv", header=True).select("Id", "PATIENT")
df_o_sp = spark.read.csv("observations.csv", header=True).select(
    "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"
)

df_merged_sp = df_o_sp.join(df_e_sp, df_o_sp.ENCOUNTER == df_e_sp.Id, "inner")
df_merged_sp = df_merged_sp.join(df_p_sp, df_o_sp.PATIENT == df_p_sp.Id, "inner")

top10_sp = (
    df_merged_sp.groupBy("DESCRIPTION")
    .count()
    .orderBy(F.col("count").desc())
    .limit(10)
    .collect()
)

t1 = time.time()
_, peak_sp = tracemalloc.get_traced_memory()
tracemalloc.stop()
spark.stop()
print(
    f"PySpark finalizado. Tiempo: {t1 - t0:.2f} s | RAM (Python driver): {peak_sp / 1024**2:.2f} MB"
)

=== ACTIVIDAD 8: BENCHMARK DE MOTORES ===

[Iniciando Pandas...]
Pandas finalizado. Tiempo: 19.10 s | RAM: 2413.14 MB

[Iniciando Polars...]


/tmp/ipykernel_548/2143982439.py:42: DeprecationWarning: `count` was renamed; use `len` instead
  top10_pl = df_merged_pl.group_by('DESCRIPTION').count().sort('count', descending=True).limit(10).collect()


Polars finalizado. Tiempo: 2.17 s | RAM (Python driver): 0.05 MB

[Iniciando PySpark...]


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/18 08:23:38 WARN Utils: Your hostname, Jorge, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/18 08:23:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/jorge/miniforge3/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/18 08:23:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Exception in thread "RemoteBlock-temp-file-clean-thread" java.la

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

In [1]:
import time
import tracemalloc

import pyspark.sql.functions as F
from pyspark.sql import SparkSession

print("=== REINTENTO PYSPARK (KERNEL LIMPIO) ===")

# Arrancamos Spark dándole 8 GB de RAM desde cero
spark = (
    SparkSession.builder.appName("Lab")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

tracemalloc.start()
t0 = time.time()

df_p_sp = spark.read.csv("patients.csv", header=True).select("Id", "RACE", "GENDER")
df_e_sp = spark.read.csv("encounters.csv", header=True).select("Id", "PATIENT")
df_o_sp = spark.read.csv("observations.csv", header=True).select(
    "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"
)

df_merged_sp = df_o_sp.join(df_e_sp, df_o_sp.ENCOUNTER == df_e_sp.Id, "inner")
df_merged_sp = df_merged_sp.join(df_p_sp, df_o_sp.PATIENT == df_p_sp.Id, "inner")

top10_sp = (
    df_merged_sp.groupBy("DESCRIPTION")
    .count()
    .orderBy(F.col("count").desc())
    .limit(10)
    .collect()
)

t1 = time.time()
_, peak_sp = tracemalloc.get_traced_memory()
tracemalloc.stop()
spark.stop()

print(
    f"PySpark finalizado. Tiempo: {t1 - t0:.2f} s | RAM (Python driver): {peak_sp / 1024**2:.2f} MB"
)

=== REINTENTO PYSPARK (KERNEL LIMPIO) ===


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/18 08:32:51 WARN Utils: Your hostname, Jorge, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/18 08:32:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/jorge/miniforge3/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/18 08:32:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                

PySpark finalizado. Tiempo: 9.26 s | RAM (Python driver): 0.73 MB


### 8. Comparativa de Motores: Pandas vs Polars vs PySpark

| Herramienta | Líneas de código | Tiempo (s) | Memoria pico (MB) | Qué costó más |
| :--- | :--- | :--- | :--- | :--- |
| **pandas** | 4 | 19.10 s | 2413.14 MB | **Consumo de RAM y Cuellos de botella:** Cargar el CSV completo en memoria para hacer cruces saturó los recursos, volviéndolo el más lento en ejecución bruta. |
| **Polars** | 4 | 2.17 s | 0.05 MB* | **Curva de aprendizaje:** Sintaxis diferente a Pandas. (*Nota: El consumo de 0.05 MB es el del driver de Python; el consumo real ocurre en Rust, fuera de la vista de `tracemalloc`). |
| **PySpark** | 5 | 9.26 s | 0.73 MB* | **El Overhead y la Configuración:** Exigió afinar la memoria a mano (8GB) para no sufrir un `OutOfMemoryError` en la JVM. Además, el tiempo de inicializar la máquina virtual de Java penaliza el rendimiento en modo local. |

**Hallazgo clave:** Polars demostró ser ~9 veces más rápido que Pandas. PySpark, aunque superó a Pandas gracias a su optimización de consultas (*Catalyst Optimizer*), no justifica su tremenda complejidad de infraestructura para correrse en un solo equipo local.

### 9. Recomendación Técnica Final

**¿Qué herramienta usaría para este volumen de datos y por qué?**
Para un volumen de datos intermedio (archivos de 1 GB a 15 GB, o ~17 millones de registros como en este caso), la recomendación es **Polars** (o herramientas equivalentes como DuckDB). 

La métricas de la Actividad 8 son concluyentes: Polars resolvió los cruces relacionales en apenas 2.17 segundos contra los 19.10 segundos de Pandas. Esto se debe a que Pandas está atado a un modelo de ejecución ansiosa (*eager evaluation*) y a un tipado basado en C/Python que asfixia la memoria RAM con metadatos. Polars, mediante Apache Arrow y su motor en Rust, implementa ejecución perezosa (*lazy evaluation*): lee el esquema, planea el cruce de tablas por adelantado y solo trae a la memoria las columnas y particiones estrictamente necesarias para la respuesta final.

**¿A partir de qué punto cambiaría de opinión?**
Mi recomendación cambiaría en dos escenarios opuestos:

1. **Si los datos bajan a < 500 MB:** Volvería a **Pandas**. Para análisis exploratorios rápidos, limpieza de strings complejas o visualización inmediata en conjuntos de datos pequeños, el ecosistema, la comunidad y la flexibilidad de Pandas siguen sin tener rival. La fricción de escribir en Pandas es mínima.
2. **Si los datos escalan a Terabytes (Big Data real):** Migraría definitivamente a **PySpark**. Si el hospital decidiera unificar historiales a nivel nacional (ej. miles de millones de observaciones pesando decenas de Terabytes), una sola máquina ya no basta. Ahí se justifica la complejidad de PySpark: su capacidad nativa para distribuir la carga de trabajo en un clúster de múltiples servidores, procesando sobre sistemas de archivos distribuidos (HDFS/S3) con tolerancia a fallos.